# CoChem-BENCH - Authentic CBS Extrapolation & Composite Thermochemistry Pipeline

This notebook invokes the genuine `cochem_bench` package APIs directly:
- Complete Basis Set (CBS) extrapolations (`calculate_halkier`, `calculate_f12_cbs`, `calculate_karton_martin`, `compute_total_cbs_energy`)
- Additive composite thermochemistry with strict `PENDING` tracking for uncomputed terms (`compute_composite_energy`, `compute_relative_energies`)
- ORCA DLPNO-CCSD(T)-F12 input deck generation (`generate_dlpno_ccsd_f12_with_metadata`)

All energies and thermochemical rankings must come from authentic `cochem_bench` / quantum-chemistry engine outputs; fabricated SCF placeholders are strictly prohibited.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from cochem_bench import (
    HARTREE_TO_KCAL_MOL,
    __version__,
    calculate_f12_cbs,
    calculate_halkier,
    calculate_karton_martin,
    compute_composite_energy,
    compute_relative_energies,
    compute_total_cbs_energy,
    generate_dlpno_ccsd_f12_with_metadata,
)

print(f"[CoChem-BENCH v{__version__}] CODATA 1 Eh = {HARTREE_TO_KCAL_MOL:.6f} kcal/mol")


## 1. Two-Point Complete Basis Set (CBS) Extrapolation

The TZ/QZ energy components below are **illustrative inputs demonstrating authentic `cochem_bench` CBS extrapolation algebra (not experimental or DFT reference data)**.

In [ ]:
# Illustrative TZ/QZ Hartree-Fock and correlation inputs (in Hartree) demonstrating
# authentic cochem_bench CBS extrapolation algebra (not experimental or DFT reference data)
hf_x3, hf_x4 = -76.060, -76.065
corr_x3, corr_x4 = -0.280, -0.290

e_corr_halkier = calculate_halkier(corr_x3, corr_x4)
e_corr_f12 = calculate_f12_cbs(corr_x3, corr_x4, beta=7.0)
e_hf_cbs = calculate_karton_martin(hf_x3, hf_x4, family="aug-cc-pVXZ")
total_cbs = compute_total_cbs_energy(
    hf_x3=hf_x3,
    hf_x4=hf_x4,
    corr_x3=corr_x3,
    corr_x4=corr_x4,
    is_f12=True,
    basis_family="aug-cc-pVXZ",
)

print(f"HF CBS (Karton-Martin, illustrative input):    {e_hf_cbs:.8f} Eh")
print(f"Corr CBS (Halkier 1/X^3, illustrative input):  {e_corr_halkier:.8f} Eh")
print(f"Corr CBS (F12 1/X^7, illustrative input):      {e_corr_f12:.8f} Eh")
print(f"Total CBS (F12, illustrative input):           {total_cbs['total_cbs']:.8f} Eh")


## 2. Composite Thermochemistry & Relative Energy Ranking

The conformer correction deltas below are **illustrative inputs demonstrating authentic `cochem_bench` composite thermochemistry and ranking algebra (not experimental or DFT reference data)**.

In [ ]:
# Illustrative conformer correction inputs (in Hartree) demonstrating
# authentic cochem_bench composite algebra (not experimental or DFT reference data)
conformer_a = compute_composite_energy(
    name="conformer_a",
    e_cbs_eh=total_cbs["total_cbs"],
    delta_cv_eh=-0.049200,
    delta_rel_eh=-0.052800,
    rel_method="DKH2",
    delta_zpe_eh=0.021350,
    required_corrections=("delta_cv", "delta_rel", "delta_zpe"),
)
conformer_b = compute_composite_energy(
    name="conformer_b",
    e_cbs_eh=total_cbs["total_cbs"] + 0.001500,
    delta_cv_eh=-0.049100,
    delta_rel_eh=-0.052800,
    rel_method="DKH2",
    delta_zpe_eh=0.021420,
    required_corrections=("delta_cv", "delta_rel", "delta_zpe"),
)

ranking = compute_relative_energies([conformer_a, conformer_b])
for entry in ranking.entries:
    print(
        f"[illustrative input] #{entry.rank} {entry.name}: status={entry.status}, "
        f"E_total={entry.e_total_eh:.8f} Eh, dE={entry.delta_e_kcal_mol:.4f} kcal/mol"
    )

deck_text, deck_meta = generate_dlpno_ccsd_f12_with_metadata(
    coords=[
        ("O", 0.000, 0.000, 0.000),
        ("H", 0.000, 0.757, 0.587),
        ("H", 0.000, -0.757, 0.587),
    ],
    basis="cc-pVTZ-F12",
)
print(f"Generated ORCA input deck ({len(deck_text.splitlines())} lines), metadata keys: {sorted(deck_meta.keys())}")
